# Kredit skoring: modelni ko'rmasdan aldash

Tasavvur qiling: bankka kredit so'rab ariza berdingiz va rad javobi oldingiz. Qarorni odam emas, model chiqargan.
Modelning ichini ko'ra olmaysiz, lekin bank API'si har bir ariza uchun 0 dan 1 gacha bo'lgan «ishonch bali»ni
qaytaradi. Ball 0.5 dan past bo'lsa rad etiladi, 0.5 va undan yuqori bo'lsa tasdiqlanadi.

Shu birgina son hujumchiga juda ko'p narsani aytib beradi. Arizadagi bitta qiymatni o'zgartirib, ball qanday
o'zgarganiga qarasangiz, model nimaga qarab qaror qilayotganini bilib olasiz. Bunday hujum **black-box evasion**
deyiladi: modelni ko'rmasdan turib, uni kerakli javobga og'dirish.

Bu darsda uch narsani o'rganasiz:

1. API qaytargan ball orqali model haqidagi ma'lumot qanday sizib chiqishini.
2. Har bir xususiyatning ta'sirini so'rovlar bilan qanday o'lchashni.
3. Cheklangan «o'zgartirish byudjeti»ni eng foydali joyga qanday sarflashni.

Python'da ro'yxat va `for` siklini bilsangiz yetarli. Mashinaviy o'rganishni bilish shart emas.

Bir gap: bu o'quv muhiti. Haqiqiy bank yoki boshqa tizimda ruxsatsiz shunday qilish jinoyat hisoblanadi.

## 1. Labga ulanamiz

Platformadagi lab sahifasini oching. U yerda **Token** degan maydon bor: undagi satrni nusxalab, pastdagi
`TOKEN` ichiga qo'ying. Token faqat sizga tegishli va bir necha soatdan keyin eskiradi. Eskirib qolsa, lab
sahifasini yangilab, yangisini oling.

In [ ]:
BASE = "https://platform.hacknow.uz/ai-poligon"
TOKEN = ""   # lab sahifasidagi Token maydonidan nusxalab, qo'shtirnoq ichiga qo'ying

assert TOKEN, "Avval TOKEN ni to'ldiring: platformadagi lab sahifasidan nusxalang."

import math
import urllib.request

urllib.request.urlretrieve(BASE + "/client/hn_lab.py", "hn_lab.py")   # labning kichik mijozi
from hn_lab import HnLab

lab = HnLab(BASE, "ml-kredit-evasion", TOKEN)
info = lab.task()
print(info["lab"]["title"])
print("Urinish:", info["attempt"], "| sarflangan so'rov:", info["queries_used"], "/", info["query_budget"])
if info["solved"]:
    print("Bu lab sizda allaqachon yechilgan. Flag:", info.get("flag"))
if info["query_budget"] - info["queries_used"] < 15:
    print("Diqqat: so'rovlar kam qoldi. Yetmay qolsa, lab.reset() bilan yangi urinish boshlang va notebook'ni boshidan yurgizing.")

## 2. Vazifa bilan tanishamiz

`lab.task()` vazifa shartini qaytaradi. Uning ichida sizning arizangiz (`x0`) bor: 8 ta son, har biri 0 bilan 1
orasida. Masalan, `oylik_daromad` 0.31 bo'lsa, bu «eng past daromad 0, eng yuqorisi 1 deb olsak, siz 0.31
dasiz» degani.

Hamma xususiyatni ham o'zgartirib bo'lmaydi. Daromad, ish staji yoki omonatni o'zgartirsa bo'ladi, lekin
yoshingizni yoki o'tgan yillardagi kechikishlaringizni o'zgartira olmaysiz. Pastdagi katak qaysi biriga tegish
mumkinligini ko'rsatadi.

In [ ]:
vazifa = info["task"]
nomlar = vazifa["features"]
x0 = vazifa["x0"]

for i, nom in enumerate(nomlar):
    izoh = "o'zgartirsa bo'ladi" if i in vazifa["mutable"] else "TEGIB BO'LMAYDI"
    print(i, nom, "=", x0[i], "—", izoh)
print()
print("O'zgartirish byudjeti:", vazifa["l1_budget"], "| tasdiqlash chegarasi:", vazifa["threshold"])

**Byudjet nima?** Hamma o'zgarishlaringiz yig'indisi byudjetdan oshmasligi kerak. Masalan, daromadni 0.30 dan
0.80 ga ko'tarsangiz 0.50 sarflaysiz. Qolgan 0.20 ni boshqa xususiyatga ishlatishingiz mumkin. Haqiqiy hayotda
ham shunday: odam o'zi haqidagi ma'lumotni cheksiz o'zgartira olmaydi.

Demak, vazifa bunday: `x0` dan boshlab shunday ariza tuzingki, tegib bo'lmaydigan xususiyatlar joyida qolsin,
jami o'zgarish byudjetdan oshmasin va model uni tasdiqlasin.

## 3. Birinchi so'rov

Avval arizani hech narsasini o'zgartirmasdan yuborib ko'ramiz. `lab.score(...)` ga qatorlar ro'yxati beriladi,
har bir qator bitta ariza.

In [ ]:
javob = lab.score({"rows": [x0]})
print(javob["output"][0])
print("Sarflangan so'rov:", javob["queries_used"], "/", javob["query_budget"])

Javobda uch narsa bor: `score` — ishonch bali (ikki xonagacha yaxlitlangan), `decision` — qaror, `cheklov` —
ariza qoidalarga to'g'ri keladimi (`ok` bo'lsa hammasi joyida).

So'rovlar cheklangan: bitta urinishda 200 ta qator yubora olasiz, bitta chaqiruvda ko'pi bilan 20 ta. Shuning
uchun har so'rovni o'ylab yuboring. So'rovlar tugab qolsa, `lab.reset()` yangi urinish boshlaydi. Lekin ehtiyot
bo'ling: reset'dan keyin ariza ham yangilanadi, ya'ni notebook'ni boshidan qayta ishga tushirish kerak bo'ladi.

## 4. Bitta xususiyatni sinab ko'ramiz

Endi asosiy g'oya. Arizadan nusxa olamiz, bitta xususiyatni o'zgartiramiz va ball qayoqqa siljiganiga qaraymiz.
Boshlanishiga daromadni eng yuqori qiymatga, ya'ni 1.0 ga qo'yamiz.

In [ ]:
sinov = list(x0)      # nusxa olamiz, asl arizaga tegmaymiz
sinov[0] = 1.0        # 0-xususiyat: oylik_daromad

javob = lab.score({"rows": [x0, sinov]})
asl, ozgargan = javob["output"]
print("asl ariza:      ", asl["score"])
print("daromad = 1.0:  ", ozgargan["score"], "| cheklov:", ozgargan["cheklov"])

Ball ko'tarildimi yoki tushdimi? Ikkalasi ham bo'lishi mumkin: bu model sun'iy, unda «daromad ko'p bo'lsa
yaxshi» degan qoida bo'lmasligi ham mumkin. Shuning uchun **taxmin qilmaymiz, o'lchaymiz**.

Yana bir narsaga e'tibor bering. Daromadni 1.0 ga qo'yganingizda `cheklov` ustunida `l1_byudjet` chiqqan
bo'lishi mumkin: bitta o'zgarishning o'zi byudjetdan oshib ketgan. Shunga qaramay model balni qaytaraverdi.
O'lchash uchun bu bizga qo'l keladi: sinov arizalari qoidaga to'g'ri kelishi shart emas, faqat eng oxirgi,
yakuniy ariza byudjet ichida bo'lishi kerak.

## 5. Topshiriq 1: hamma xususiyatlarni o'lchang

Endi shu ishni o'zgartirsa bo'ladigan har bir xususiyat uchun qilasiz. Har biri uchun ikkita sinov kerak:
qiymatni 0.0 ga qo'yib ko'rish va 1.0 ga qo'yib ko'rish. Beshta xususiyat bor, demak o'nta sinov.

Pastdagi katakka `for` sikli yozing. Har qadamda xuddi 4-bo'limdagidek `x0` dan nusxa oling, kerakli
xususiyatga qiymatni qo'ying va natijani `sinovlar` ro'yxatiga uchlik qilib qo'shing:
`(xususiyat raqami, qo'yilgan qiymat, ariza)`.

Yordam: xususiyat raqamlari `vazifa["mutable"]` ichida turibdi, qiymatlar esa `(0.0, 1.0)`.

In [ ]:
sinovlar = []   # har element: (xususiyat raqami, qo'yilgan qiymat, ariza)

# TOPSHIRIQ 1: siklni shu yerga yozing.

Keyingi katak avval ro'yxatingizni tekshiradi (so'rov bekorga ketmasligi uchun), keyin hamma sinovni bitta
chaqiruvda yuboradi va natijani tartiblab chiqaradi.

Natijani solishtirishda bitta nozik joy bor. Ball 0 bilan 1 orasiga «siqilgan»: 0.90 dan 0.95 ga chiqish 0.50
dan 0.55 ga chiqishdan ancha qiyin. Adolatli solishtirish uchun balni `logit` degan siqilmagan shkalaga
o'tkazamiz. Funksiyasi tayyor turibdi, formulasini yodlash shart emas.

In [ ]:
def logit(ball):
    ball = min(max(ball, 0.005), 0.995)   # 0 yoki 1 chiqib qolsa, cheksizlikka ketib qolmasin
    return math.log(ball / (1 - ball))


def sinovlar_togri(royxat):
    if len(royxat) != 2 * len(vazifa["mutable"]):
        print("Hali tayyor emas: ro'yxatda", len(royxat), "ta sinov bor, kerak:", 2 * len(vazifa["mutable"]))
        return False
    for i, qiymat, ariza in royxat:
        togri = (len(ariza) == len(x0) and i in vazifa["mutable"] and qiymat in (0.0, 1.0) and ariza[i] == qiymat
                 and all(ariza[k] == x0[k] for k in range(len(x0)) if k != i))
        if not togri:
            print("Bu sinovda xato bor:", (i, qiymat), "— arizada faqat shu bitta xususiyat o'zgarishi kerak.")
            return False
    return True


yollar = []
if sinovlar_togri(sinovlar):
    javob = lab.score({"rows": [x0] + [ariza for _, _, ariza in sinovlar]})
    asl_ball = javob["output"][0]["score"]
    for (i, qiymat, _), natija in zip(sinovlar, javob["output"][1:]):
        narx = abs(qiymat - x0[i])
        foyda = logit(natija["score"]) - logit(asl_ball)
        yollar.append((foyda / narx, nomlar[i], i, qiymat, narx, foyda))
    yollar.sort(reverse=True)
    for samara, nom, i, qiymat, narx, foyda in yollar:
        print("%d %-22s -> %.1f | narx %.2f | foyda %+.2f | samara %+.2f" % (i, nom, qiymat, narx, foyda, samara))
    print("Sarflangan so'rov:", javob["queries_used"], "/", javob["query_budget"])

Har satr bitta «yo'l»ni ko'rsatadi: qaysi xususiyatni qaysi tomonga surish.

- **narx** — shu yo'lni oxirigacha bossangiz byudjetdan qancha ketadi;
- **foyda** — ball (logit shkalasida) qanchaga ko'tariladi. Manfiy bo'lsa, bu yo'l zarar qiladi;
- **samara** — bir birlik byudjetga qancha foyda to'g'ri keladi. Ro'yxat shu ustun bo'yicha tartiblangan.

Har bir xususiyat ikki marta uchraydi (0.0 tomonga va 1.0 tomonga), ulardan faqat bittasi foydali bo'ladi.

## 6. Topshiriq 2: byudjetni taqsimlang

Byudjet oz, shuning uchun uni eng samarali yo'llarga sarflaymiz. Qoida oddiy: ro'yxatning tepasidan boshlang,
yo'lni iloji boricha oxirigacha bosing, byudjet tugaguncha keyingisiga o'ting.

Kichik misol (raqamlar o'ylab topilgan). Byudjet 0.7. A yo'lining samarasi 3.0, narxi 0.4. B yo'lining
samarasi 2.0, narxi 0.5. Avval A ni to'liq olamiz: 0.4 ketdi, 0.3 qoldi. Keyin B keladi, unga 0.5 kerak, bizda
esa 0.3 bor. Demak, B ni oxirigacha emas, 0.3 ga suramiz. Bu modelda yo'lning bir qismi foydaning ham shuncha
qismini beradi.

Pastdagi katakda `yakuniy` arizani tuzing. To'liq surish uchun qiymatni 0.0 yoki 1.0 ga qo'yasiz, qisman
surish uchun `x0` dagi qiymatga kerakli miqdorni qo'shasiz yoki ayirasiz.

In [ ]:
yakuniy = list(x0)

# TOPSHIRIQ 2: jadvalga qarab yakuniy arizani tuzing. Namuna (raqamlarni o'zingizniki bilan almashtiring):
# yakuniy[0] = 1.0            # 0-xususiyatni oxirigacha surish
# yakuniy[3] = x0[3] - 0.2    # 3-xususiyatni 0.2 ga kamaytirish

Yuborishdan oldin arizani o'zimiz tekshirib olamiz. Bu tekshiruv so'rov sarflamaydi.

In [ ]:
def tekshir(ariza):
    if len(ariza) != len(x0):
        print("Xato: arizada", len(x0), "ta son bo'lishi kerak.")
        return False
    sarf = sum(abs(a - b) for a, b in zip(ariza, x0))
    print("Sarflangan byudjet: %.3f / %.1f" % (sarf, vazifa["l1_budget"]))
    if any(ariza[i] != x0[i] for i in vazifa["immutable"]):
        print("Xato: tegib bo'lmaydigan xususiyat o'zgargan.")
        return False
    if any(not 0.0 <= qiymat <= 1.0 for qiymat in ariza):
        print("Xato: hamma qiymat 0 bilan 1 orasida bo'lishi kerak.")
        return False
    if sarf > vazifa["l1_budget"] + 1e-9:
        print("Xato: byudjetdan oshib ketdi. Oxirgi yo'lni biroz qisqartiring.")
        return False
    return True


if yakuniy == list(x0):
    print("Hali hech narsa o'zgartirmadingiz: Topshiriq 2 ni bajaring.")
elif tekshir(yakuniy):
    javob = lab.score({"rows": [yakuniy]})
    natija = javob["output"][0]
    print(natija)
    if javob["solved"]:
        print("Bo'ldi! Model arizani tasdiqladi.")
        print("Flag:", javob["flag"])
    else:
        print("Hali tasdiqlanmadi. Ball 0.5 ga yetishi kerak.")
        print("Byudjetdan qolganini ham ishlating va eng samarali yo'llarni to'liq bosganingizni tekshiring.")

Flag chiqqan bo'lsa, uni platformadagi topshiriq sahifasiga yuboring. Flag faqat sizniki: boshqa odamning
flag'i sizda ishlamaydi.

## 7. Nima bo'ldi?

Siz modelning ichini ko'rmadingiz, kodini o'qimadingiz. Bor-yo'g'i o'n beshtacha so'rov yubordingiz va model
nimani yoqtirishini, nimani yoqtirmasligini bilib oldingiz. Sababi bitta: API qarordan tashqari ishonch balini
ham qaytaradi. Balni ikki xonagacha yaxlitlash ham yordam bermadi.

Himoyachi nima qilishi mumkin? Bir nechta yo'l bor, lekin hech biri sehrli tayoqcha emas. Birinchisi, balni
umuman qaytarmaslik va faqat «tasdiqlandi» yoki «rad etildi» deyish. Bu hujumni qiyinlashtiradi, lekin
to'xtatmaydi: faqat qarorga qarab ham chegarani topsa bo'ladi, shunchaki ko'proq so'rov ketadi. Ikkinchisi,
so'rovlar sonini cheklash va g'alati so'rovlarni kuzatish: bitta odamning arizasi bir daqiqada o'n marta
o'zgarib, qiymatlari goh 0, goh 1 bo'lib tursa, bu oddiy mijozga o'xshamaydi. Uchinchisi, arizadagi
ma'lumotni hujjat bilan tekshirish: daromadni yozib qo'yish oson, tasdiqlash qiyin.

## Qo'shimcha (xohlasangiz)

1. Topshiriq 2 ni qo'lda emas, kod bilan bajaring: `yollar` ro'yxatidan o'zi yakuniy arizani tuzadigan sikl yozing.
2. So'rovlar sonini kamaytirib ko'ring. Har xususiyat uchun ikkita sinov shartmi?
3. O'ylab ko'ring: API faqat qarorni qaytarganda bu hujumni qanday qilgan bo'lardingiz?